# LeetCode 947 — Most Stones Removed with Same Row or Column bằng Python

Link đề: https://leetcode.com/problems/most-stones-removed-with-same-row-or-column/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Union Find, Graph, Hash Table

Skeleton LeetCode:

```python
class Solution:
    def removeStones(self, stones: List[List[int]]) -> int:
        ...
```

Ghi chú: đáp án duy nhất (một số nguyên). Điểm cốt lõi: hai viên đá **liên thông** nếu cùng hàng hoặc cùng cột; trong **mỗi thành phần liên thông** có thể loại bỏ **tất cả trừ một** viên (bỏ theo thứ tự DFS ngược, lá trước); và không thể loại viên cuối cùng của một thành phần (nó không còn bạn cùng hàng/cột). Vậy đáp án `= n − (số thành phần liên thông)`. Oracle: **quy hoạch động trên tập con** (bitmask) mô phỏng đúng luật loại bỏ, không giả định kết quả về thành phần liên thông.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một viên đá được loại bỏ nếu còn viên đá khác (chưa bị loại) **cùng hàng hoặc cùng cột**; tìm số viên **nhiều nhất** có thể loại.
2. Nắm công cụ Python: đồ thị ngầm định, DFS/BFS lặp, **Union-Find** trên các nút "hàng" và "cột", `dict` nhóm theo toạ độ, bitmask DP để làm oracle.
3. Đi từ **đồ thị đá + DFS (`O(n²)`)** → **Union-Find trên hàng/cột (`O(n α)`)** → **BFS theo bảng băm hàng/cột** → **xây dãy loại bỏ tường minh bằng DFS ngược**.
4. Hiểu mấu chốt: đáp án `= n − C` với `C` là số thành phần liên thông của đồ thị mà đỉnh là đá (cạnh: cùng hàng/cột); tương đương đồ thị **hai phía** hàng–cột (mỗi viên đá là một cạnh nối hàng `x` với cột `y`).
5. Tránh lỗi kinh điển: trả số thành phần thay vì `n − C`, gộp **hàng và cột chung một không gian chỉ số** (`x` và `y` va chạm), chỉ nối theo hàng mà quên cột, đếm số hàng/cột phân biệt thay cho thành phần, và đệ quy DFS quá sâu.

## 2. Đề bài & ví dụ

Trên mặt phẳng toạ độ có `n` viên đá đặt tại các điểm nguyên (mỗi điểm tối đa một viên).

Một viên đá có thể bị **loại bỏ** nếu nó cùng **hàng** hoặc cùng **cột** với **một viên đá khác chưa bị loại**.

Cho mảng `stones` độ dài `n`, `stones[i] = [xi, yi]` là vị trí viên đá thứ `i`. Trả về số viên đá **tối đa** có thể loại bỏ.

**Ví dụ 1:** `stones = [[0,0],[0,1],[1,0],[1,2],[2,1],[2,2]]` → `5` (chỉ còn lại `[0,0]`).

**Ví dụ 2:** `stones = [[0,0],[0,2],[1,1],[2,0],[2,2]]` → `3` (còn lại `[0,0]` và `[1,1]`).

**Ví dụ 3:** `stones = [[0,0]]` → `0`.

**Ràng buộc:** `1 <= n <= 1000`; `0 <= xi, yi <= 10^4`; không có hai viên đá cùng một điểm.

In [1]:
import itertools
import random
import sys
import time
from collections import defaultdict, deque
from typing import List

examples = [
    ([[0, 0], [0, 1], [1, 0], [1, 2], [2, 1], [2, 2]], 5),
    ([[0, 0], [0, 2], [1, 1], [2, 0], [2, 2]], 3),
    ([[0, 0]], 0),
]
for st, want in examples:
    print(st, "->", want)

[[0, 0], [0, 1], [1, 0], [1, 2], [2, 1], [2, 2]] -> 5
[[0, 0], [0, 2], [1, 1], [2, 0], [2, 2]] -> 3
[[0, 0]] -> 0


## 3. Kiến thức nền

### 3.1 Đồ thị đá và thành phần liên thông

Đồ thị `G`: mỗi viên đá là một đỉnh; hai đỉnh nối cạnh nếu **cùng hàng** hoặc **cùng cột**. Một **thành phần liên thông** là tập đá mà từ viên này đi tới viên kia qua chuỗi các bước "cùng hàng/cột".

In [2]:
def components_pairs(stones):
    n = len(stones)
    adj = [[j for j in range(n) if j != i and (stones[i][0] == stones[j][0] or stones[i][1] == stones[j][1])] for i in range(n)]
    seen, comps = [False] * n, []
    for s in range(n):
        if not seen[s]:
            stack, comp = [s], []
            seen[s] = True
            while stack:
                u = stack.pop()
                comp.append(u)
                for v in adj[u]:
                    if not seen[v]:
                        seen[v] = True
                        stack.append(v)
            comps.append(sorted(comp))
    return comps


print(components_pairs(examples[0][0]), "|", components_pairs(examples[1][0]))

[[0, 1, 2, 3, 4, 5]] | [[0, 1, 3, 4], [2]]


### 3.2 Đồ thị hai phía hàng–cột

Thay vì đồ thị `n` đỉnh có tới `O(n²)` cạnh, dựng đồ thị có đỉnh là **hàng** và **cột**: viên đá `(x, y)` là **một cạnh** nối đỉnh hàng `x` với đỉnh cột `y`. Hai viên đá cùng thành phần ⇔ cạnh của chúng cùng thành phần liên thông trong đồ thị hai phía. Chỉ `n` cạnh, tối đa `2n` đỉnh.

### 3.3 Union-Find (DSU)

`find(a)` trả đại diện của tập chứa `a` (nén đường đi), `union(a, b)` gộp hai tập. Với `n` cạnh: `O(n α(n))`. Cần **tách không gian chỉ số** hàng và cột (ví dụ `("r", x)` và `("c", y)` hoặc `y + 10001`).

### 3.4 Vì sao `n − C`

Trong mỗi thành phần liên thông `k` viên: có thể loại `k − 1` viên (xem mục 10); viên cuối cùng không loại được. Tổng loại tối đa `= Σ (k_c − 1) = n − C`.

In [3]:
class DSU:
    def __init__(self):
        self.parent = {}

    def find(self, a):
        self.parent.setdefault(a, a)
        root = a
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[a] != root:                     # nén đường đi
            self.parent[a], a = root, self.parent[a]
        return root

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb


d = DSU()
for x, y in examples[1][0]:
    d.union(("r", x), ("c", y))                           # tách không gian chỉ số hàng/cột
print("số thành phần (đồ thị hàng-cột):", len({d.find(("r", x)) for x, y in examples[1][0]}))

số thành phần (đồ thị hàng-cột): 2


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2: `[[0,0],[0,2],[1,1],[2,0],[2,2]]`.

```
y:   0 1 2
x=0  ● . ●
x=1  . ● .
x=2  ● . ●
```

Bốn đá ở bốn góc nối với nhau (cùng hàng `0`: `[0,0]-[0,2]`; cùng cột `2`: `[0,2]-[2,2]`; ...) tạo **một** thành phần `k = 4`; viên `[1,1]` đứng một mình (không chung hàng/cột với viên nào) là thành phần `k = 1`. Số thành phần `C = 2`, đáp án `5 − 2 = 3` ✓ (loại ba viên của nhóm góc, còn lại một viên góc và `[1,1]`).

In [4]:
for st, want in examples:
    comps = components_pairs(st)
    print(f"n = {len(st)}: thành phần {comps} -> n - C = {len(st) - len(comps)} (đề: {want})")
    assert len(st) - len(comps) == want

n = 6: thành phần [[0, 1, 2, 3, 4, 5]] -> n - C = 5 (đề: 5)
n = 5: thành phần [[0, 1, 3, 4], [2]] -> n - C = 3 (đề: 3)
n = 1: thành phần [[0]] -> n - C = 0 (đề: 0)


## 5. Cách 1 — Đồ thị đá + DFS

**Ý tưởng:** dựng danh sách kề trực tiếp giữa các viên đá (hai đá kề nhau nếu cùng hàng hoặc cột): `O(n²)`; đếm thành phần liên thông bằng DFS lặp; đáp án `n − C`.

In [5]:
def remove_stones_dfs(stones):
    n = len(stones)
    seen = [False] * n
    comps = 0
    for s in range(n):
        if seen[s]:
            continue
        comps += 1                                       # gặp một đá chưa thăm: một thành phần mới
        stack = [s]
        seen[s] = True
        while stack:                                     # DFS lặp: tránh giới hạn đệ quy
            u = stack.pop()
            for v in range(n):
                if not seen[v] and (stones[u][0] == stones[v][0] or stones[u][1] == stones[v][1]):
                    seen[v] = True
                    stack.append(v)
    return n - comps


for st, want in examples:
    assert remove_stones_dfs(st) == want
print([remove_stones_dfs(st) for st, _ in examples])

[5, 3, 0]


In [6]:
# Code nộp LeetCode cho Cách 1 (O(n^2))
class SolutionDFS:
    def removeStones(self, stones: List[List[int]]) -> int:
        n = len(stones)
        seen = [False] * n
        comps = 0
        for s in range(n):
            if seen[s]:
                continue
            comps += 1
            stack = [s]
            seen[s] = True
            while stack:
                u = stack.pop()
                for v in range(n):
                    if not seen[v] and (stones[u][0] == stones[v][0] or stones[u][1] == stones[v][1]):
                        seen[v] = True
                        stack.append(v)
        return n - comps

**Độ phức tạp:** `O(n²)` thời gian (với `n ≤ 1000`: `10^6` phép), `O(n)` bộ nhớ.

**Nút thắt:** so cặp đá `O(n²)`; chỉ cần nhóm theo hàng và cột.

## 6. Ý tưởng thuật toán / bước đột phá

Coi mỗi viên đá là **cạnh** nối nút-hàng `x` với nút-cột `y`. Số thành phần của đá = số thành phần (chứa cạnh) của đồ thị hai phía. Dùng **Union-Find** với `n` phép `union(("r", x), ("c", y))`; đếm số tập khác nhau; đáp án `n − C`.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** Union-Find trên hàng/cột, **Cách 3** BFS với bảng băm hàng/cột, **Cách 4** xây dãy loại bỏ tường minh.

### Cách 2 — Union-Find trên nút hàng và nút cột

**Giải thích từng đoạn:**

1. Với mỗi đá `(x, y)`: `union(("r", x), ("c", y))`.
2. Số thành phần `C` = số gốc khác nhau trong tập `{find(("r", x))}` của các đá.
3. Đáp án `n − C`.

In [7]:
def remove_stones_dsu(stones):
    parent = {}

    def find(a):
        parent.setdefault(a, a)
        root = a
        while parent[root] != root:
            root = parent[root]
        while parent[a] != root:
            parent[a], a = root, parent[a]
        return root

    for x, y in stones:
        ra, rb = find(("r", x)), find(("c", y))          # hàng và cột ở hai "không gian" khác nhau
        if ra != rb:
            parent[ra] = rb
    comps = len({find(("r", x)) for x, y in stones})     # mỗi đá thuộc thành phần của hàng của nó
    return len(stones) - comps


for st, want in examples:
    assert remove_stones_dsu(st) == want
print([remove_stones_dsu(st) for st, _ in examples])

[5, 3, 0]


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionDSU:
    def removeStones(self, stones: List[List[int]]) -> int:
        parent = {}

        def find(a):
            parent.setdefault(a, a)
            root = a
            while parent[root] != root:
                root = parent[root]
            while parent[a] != root:
                parent[a], a = root, parent[a]
            return root

        for x, y in stones:
            ra, rb = find(("r", x)), find(("c", y))
            if ra != rb:
                parent[ra] = rb
        return len(stones) - len({find(("r", x)) for x, y in stones})

**Độ phức tạp:** `O(n α(n))` thời gian, `O(n)` bộ nhớ. Cách chuẩn.

### Cách 3 — BFS với bảng băm hàng/cột

**Ý tưởng:** `rows[x]` = danh sách chỉ số đá ở hàng `x`; `cols[y]` tương tự. BFS từ một đá chưa thăm: lấy mọi đá cùng hàng và cùng cột (qua bảng băm, **không** quét `n`), đánh dấu **cả hàng và cột đã dùng** để không duyệt lại danh sách. Mỗi thành phần đếm một lần; đáp án `n − C`.

In [9]:
def remove_stones_bfs(stones):
    rows, cols = defaultdict(list), defaultdict(list)
    for i, (x, y) in enumerate(stones):
        rows[x].append(i)
        cols[y].append(i)
    seen = [False] * len(stones)
    used_row, used_col = set(), set()
    comps = 0
    for s in range(len(stones)):
        if seen[s]:
            continue
        comps += 1
        q = deque([s])
        seen[s] = True
        while q:
            u = q.popleft()
            x, y = stones[u]
            if x not in used_row:                        # mỗi hàng/cột chỉ cần duyệt danh sách MỘT lần
                used_row.add(x)
                for v in rows[x]:
                    if not seen[v]:
                        seen[v] = True
                        q.append(v)
            if y not in used_col:
                used_col.add(y)
                for v in cols[y]:
                    if not seen[v]:
                        seen[v] = True
                        q.append(v)
    return len(stones) - comps


for st, want in examples:
    assert remove_stones_bfs(st) == want
print([remove_stones_bfs(st) for st, _ in examples])

[5, 3, 0]


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionBFS:
    def removeStones(self, stones: List[List[int]]) -> int:
        rows, cols = defaultdict(list), defaultdict(list)
        for i, (x, y) in enumerate(stones):
            rows[x].append(i)
            cols[y].append(i)
        seen = [False] * len(stones)
        used_row, used_col = set(), set()
        comps = 0
        for s in range(len(stones)):
            if seen[s]:
                continue
            comps += 1
            q = deque([s])
            seen[s] = True
            while q:
                u = q.popleft()
                x, y = stones[u]
                if x not in used_row:
                    used_row.add(x)
                    for v in rows[x]:
                        if not seen[v]:
                            seen[v] = True
                            q.append(v)
                if y not in used_col:
                    used_col.add(y)
                    for v in cols[y]:
                        if not seen[v]:
                            seen[v] = True
                            q.append(v)
        return len(stones) - comps

**Độ phức tạp:** `O(n)` thời gian (mỗi danh sách hàng/cột duyệt một lần, mỗi đá vào hàng đợi một lần), `O(n)` bộ nhớ.

### Cách 4 — Xây dãy loại bỏ tường minh (DFS ngược)

**Ý tưởng:** chứng minh bằng **hành động**: với mỗi thành phần chọn gốc, duyệt DFS lấy thứ tự thăm; loại bỏ các đá **theo thứ tự thăm ngược** (đỉnh thăm sau loại trước), giữ lại gốc. Hàm kiểm tra mô phỏng từng bước: mỗi viên bị loại phải còn một viên khác cùng hàng/cột chưa bị loại. Đáp án = độ dài dãy hợp lệ (và ta kiểm chứng tính hợp lệ).

In [11]:
def removal_sequence(stones):
    """Trả danh sách chỉ số đá theo thứ tự loại bỏ (mỗi thành phần: thứ tự thăm DFS ngược, giữ lại gốc)."""
    n = len(stones)
    rows, cols = defaultdict(list), defaultdict(list)
    for i, (x, y) in enumerate(stones):
        rows[x].append(i)
        cols[y].append(i)
    seen = [False] * n
    seq = []
    for s in range(n):
        if seen[s]:
            continue
        order, stack = [], [s]
        seen[s] = True
        while stack:
            u = stack.pop()
            order.append(u)                              # thứ tự thăm (tiền thứ tự)
            x, y = stones[u]
            for v in rows[x] + cols[y]:
                if not seen[v]:
                    seen[v] = True
                    stack.append(v)
        seq.extend(reversed(order[1:]))                  # loại theo thứ tự thăm NGƯỢC, giữ lại gốc order[0]
    return seq


def simulate_removals(stones, seq):
    """Kiểm tra mỗi viên bị loại còn một viên khác (chưa bị loại) cùng hàng hoặc cùng cột."""
    alive = set(range(len(stones)))
    for i in seq:
        assert i in alive
        x, y = stones[i]
        assert any(j != i and (stones[j][0] == x or stones[j][1] == y) for j in alive), ("không hợp lệ", i)
        alive.remove(i)
    return True


def remove_stones_sequence(stones):
    seq = removal_sequence(stones)
    simulate_removals(stones, seq)                       # dãy hợp lệ: chứng minh đạt được
    return len(seq)


for st, want in examples:
    assert remove_stones_sequence(st) == want
print([remove_stones_sequence(st) for st, _ in examples], "| dãy ví dụ 2:", removal_sequence(examples[1][0]))

[5, 3, 0] | dãy ví dụ 2: [1, 4, 3]


In [12]:
# Code nộp LeetCode cho Cách 4 (đếm độ dài dãy loại bỏ; không cần mô phỏng kiểm tra khi nộp)
class SolutionSequence:
    def removeStones(self, stones: List[List[int]]) -> int:
        rows, cols = defaultdict(list), defaultdict(list)
        for i, (x, y) in enumerate(stones):
            rows[x].append(i)
            cols[y].append(i)
        seen = [False] * len(stones)
        removed = 0
        for s in range(len(stones)):
            if seen[s]:
                continue
            size, stack = 0, [s]
            seen[s] = True
            while stack:
                u = stack.pop()
                size += 1
                x, y = stones[u]
                for v in rows[x] + cols[y]:
                    if not seen[v]:
                        seen[v] = True
                        stack.append(v)
            removed += size - 1                          # mỗi thành phần loại được size - 1 viên
        return removed

**Độ phức tạp:** `O(n)` thời gian cho dựng dãy; (mô phỏng kiểm tra `O(n²)` chỉ dùng trong test). Cách này **chứng minh** cận dưới `n − C` bằng cấu trúc.

## 8. TRACE từng bước

**Trace Cách 2** với ví dụ 2 `[[0,0],[0,2],[1,1],[2,0],[2,2]]`:

| đá `(x, y)` | `union` | các tập sau |
|---|---|---|
| `(0,0)` | `r0 ∪ c0` | `{r0,c0}` |
| `(0,2)` | `r0 ∪ c2` | `{r0,c0,c2}` |
| `(1,1)` | `r1 ∪ c1` | `{r1,c1}` |
| `(2,0)` | `r2 ∪ c0` | `{r0,c0,c2,r2}` |
| `(2,2)` | `r2 ∪ c2` (đã chung) | không đổi |

Hai tập ⇒ `C = 2` ⇒ `5 − 2 = 3` ✓. Hàm trace in tập sau mỗi bước:

In [13]:
def trace_dsu(stones):
    parent = {}

    def find(a):
        parent.setdefault(a, a)
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def groups():
        g = defaultdict(list)
        for node in list(parent):
            g[find(node)].append(node)
        return sorted(sorted(v) for v in g.values())

    for x, y in stones:
        a, b = find(("r", x)), find(("c", y))
        merged = a != b
        if merged:
            parent[a] = b
        print(f"đá ({x},{y}): union(r{x}, c{y}) {'gộp' if merged else '(đã chung)'} -> {[[f'{t}{v}' for t, v in grp] for grp in groups()]}")
    comps = len({find(("r", x)) for x, y in stones})
    print(f"C = {comps}; đáp án = {len(stones)} - {comps} = {len(stones) - comps}")
    return len(stones) - comps


assert trace_dsu(examples[1][0]) == 3

đá (0,0): union(r0, c0) gộp -> [['c0', 'r0']]
đá (0,2): union(r0, c2) gộp -> [['c0', 'c2', 'r0']]
đá (1,1): union(r1, c1) gộp -> [['c0', 'c2', 'r0'], ['c1', 'r1']]
đá (2,0): union(r2, c0) gộp -> [['c0', 'c2', 'r0', 'r2'], ['c1', 'r1']]
đá (2,2): union(r2, c2) (đã chung) -> [['c0', 'c2', 'r0', 'r2'], ['c1', 'r1']]
C = 2; đáp án = 5 - 2 = 3


**Trace Cách 4:** thành phần `{0,1,3,4}` (bốn đá góc), gốc `0`; thứ tự thăm DFS có thể là `0, 3, 4, 1`...; loại ngược `1, 4, 3` (giữ `0`): mỗi viên bị loại còn một viên cùng hàng/cột (đá thăm trước nó, chưa bị loại). Đá `[1,1]` (thành phần riêng) không loại. Tổng `3`.

## 9. Code nộp LeetCode

Chọn **Cách 2** (Union-Find trên hàng/cột). Giữ nguyên `class Solution`, `removeStones(self, stones)`:

In [14]:
class Solution:
    def removeStones(self, stones: List[List[int]]) -> int:
        parent = {}                                        # Union-Find trên các nút "hàng" và "cột"

        def find(a):
            parent.setdefault(a, a)
            root = a
            while parent[root] != root:
                root = parent[root]
            while parent[a] != root:                       # nén đường đi
                parent[a], a = root, parent[a]
            return root

        for x, y in stones:                                # mỗi viên đá nối hàng x với cột y
            ra, rb = find(("r", x)), find(("c", y))        # tách không gian hàng/cột (x và y có thể trùng số)
            if ra != rb:
                parent[ra] = rb
        components = len({find(("r", x)) for x, y in stones})
        return len(stones) - components                    # mỗi thành phần chỉ giữ lại một viên


print([Solution().removeStones([list(p) for p in st]) for st, _ in examples])      # [5, 3, 0]
assert all(Solution().removeStones([list(p) for p in st]) == want for st, want in examples)

[5, 3, 0]


## 10. Vì sao đúng

**(a) Cận trên: viên cuối của mỗi thành phần không loại được.** Xét một thành phần liên thông `K` của đồ thị đá. Trong quá trình loại bỏ, một viên chỉ bị loại khi còn một viên **chưa loại** cùng hàng/cột, tức **cùng thành phần** với nó (vì hai viên cùng hàng/cột thuộc cùng thành phần). Vì vậy không thể loại viên **duy nhất còn lại** của `K` (không còn viên nào cùng hàng/cột trong `K`; các viên ở thành phần khác không chung hàng/cột với nó). Do đó mỗi thành phần phải còn **ít nhất một** viên ⇒ số loại tối đa `≤ n − C`.

**(b) Cận dưới: loại được `k − 1` viên của mỗi thành phần `k` viên.** Lấy một **cây khung** của thành phần (DFS từ một gốc `r`); đánh số các đỉnh theo thứ tự thăm `v₀ = r, v₁, v₂, …`. Mỗi đỉnh `v_i` (`i ≥ 1`) có **cha** thăm trước nó (chỉ số nhỏ hơn) và cha cùng hàng/cột với `v_i`. Loại các đỉnh theo thứ tự **giảm dần chỉ số** `v_{k−1}, …, v_1`: khi tới lượt `v_i`, cha của nó (chỉ số `< i`) **chưa bị loại**, và cùng hàng/cột ⇒ loại hợp lệ. Còn lại gốc `v₀`. Vậy loại được `k − 1` viên mỗi thành phần ⇒ tổng `n − C`.

**(c) Hai đồ thị cho cùng số thành phần.** Với đồ thị hai phía hàng–cột (đỉnh: hàng, cột; cạnh: đá), hai đá cùng thành phần của đồ thị đá ⇔ chúng cùng thành phần trong đồ thị hai phía: đá `(x₁, y₁)` và `(x₂, y₂)` cùng hàng (`x₁ = x₂`) ⇔ hai cạnh cùng chung đỉnh `r_x`; tương tự cho cột; và liên thông bắc cầu giữ nguyên. Số thành phần của đá = số thành phần (có cạnh) của đồ thị hai phía = số gốc khác nhau của `find(("r", x))`.

**(d) Tách không gian chỉ số.** Nếu dùng cùng số nguyên cho hàng và cột (`union(x, y)`), hàng `x = 1` và cột `y = 1` bị coi là **một** nút, tạo liên thông giả (ví dụ `(0, 1)` và `(1, 2)` không chung hàng/cột nhưng bị nối qua nút `1`). Dùng `("r", x)` và `("c", y)` (hoặc `y + 10001`).

**(e) Cách 3.** BFS theo bảng băm duyệt mỗi danh sách hàng/cột đúng một lần và đánh dấu đã thăm nên mỗi đá vào hàng đợi một lần; số lần BFS khởi tạo = số thành phần.

**(f) Cách 4.** Chính là chứng minh (b) bằng mã: thứ tự thăm DFS (tiền thứ tự) và loại theo thứ tự ngược; hàm `simulate_removals` kiểm tra tính hợp lệ từng bước.

Ta kiểm chứng bằng máy: mọi cách khớp oracle **DP tập con** (luật loại bỏ mô phỏng nguyên văn) trên **mọi** tập con của lưới `3 × 3` và `3 × 4` (mục 12D) và ngẫu nhiên (12E); và khớp nhau ở `n = 1000` (12F).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Đồ thị đá + DFS | `O(n²)` | `O(n)` | `n ≤ 1000` vẫn chạy được |
| 2. Union-Find hàng/cột | `O(n α(n))` | `O(n)` | **đáp án chính** |
| 3. BFS với bảng băm | `O(n)` | `O(n)` | không cần DSU |
| 4. Dãy loại bỏ tường minh | `O(n)` (+ kiểm tra `O(n²)`) | `O(n)` | chứng minh cận dưới |

Đo thực tế (`n = 1000`):

In [15]:
def bench(fn, arg, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(arg)
        best = min(best, time.perf_counter() - t0)
    return best


def random_stones(rng, n, X, Y):
    pts = set()
    while len(pts) < n:
        pts.add((rng.randint(0, X), rng.randint(0, Y)))
    return [list(p) for p in pts]


rng = random.Random(947)
for label, (X, Y) in [("thưa 10^4", (10 ** 4, 10 ** 4)), ("đặc 40x40", (39, 39)), ("dải 3x400", (2, 399))]:
    st = random_stones(rng, 1000, X, Y)
    row = [f"{name} {bench(fn, st) * 1e3:8.2f} ms" for name, fn in
           [("dfs", remove_stones_dfs), ("dsu", remove_stones_dsu), ("bfs", remove_stones_bfs)]]
    print(f"{label:12s}: " + " | ".join(row))

thưa 10^4   : dfs    64.97 ms | dsu     1.19 ms | bfs     1.24 ms
đặc 40x40   : dfs    31.89 ms | dsu     1.42 ms | bfs     0.39 ms
dải 3x400   : dfs    26.91 ms | dsu     1.08 ms | bfs     0.34 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi tập con của lưới nhỏ với oracle DP tập con, (E) ngẫu nhiên có seed + tính chất, (F) giới hạn (`n = 1000`).

### A. Ví dụ của đề

In [16]:
def oracle(stones):
    """Oracle: DP trên tập con còn lại (bitmask), mô phỏng đúng luật: một viên bị loại nếu còn viên khác CÙNG HÀNG hoặc CÙNG CỘT
    chưa bị loại. best[mask] = số viên tối đa còn loại được khi tập còn lại là mask. Không giả định gì về thành phần liên thông."""
    n = len(stones)
    adj = [0] * n
    for i in range(n):
        for j in range(n):
            if i != j and (stones[i][0] == stones[j][0] or stones[i][1] == stones[j][1]):
                adj[i] |= 1 << j
    best = [0] * (1 << n)
    for mask in range(1, 1 << n):
        b = 0
        m = mask
        while m:
            low = m & -m
            s = low.bit_length() - 1
            m ^= low
            if adj[s] & mask:                            # còn viên khác (trong mask) cùng hàng/cột: có thể loại viên s
                b = max(b, 1 + best[mask ^ low])
        best[mask] = b
    return best[(1 << n) - 1]


IMPLS = {
    "dfs": remove_stones_dfs,
    "dsu": remove_stones_dsu,
    "bfs": remove_stones_bfs,
    "sequence": remove_stones_sequence,
    "Solution": lambda st: Solution().removeStones([list(p) for p in st]),
    "SolutionDFS": lambda st: SolutionDFS().removeStones([list(p) for p in st]),
    "SolutionDSU": lambda st: SolutionDSU().removeStones([list(p) for p in st]),
    "SolutionBFS": lambda st: SolutionBFS().removeStones([list(p) for p in st]),
    "SolutionSequence": lambda st: SolutionSequence().removeStones([list(p) for p in st]),
}


def check(stones, want=None, names=None):
    if want is None:
        want = oracle(stones)
    for name in (names or IMPLS):
        got = IMPLS[name](stones)
        assert got == want, (name, stones, got, want)


for st, want in examples:
    assert oracle(st) == want
    check(st, want)
print("ví dụ đề: oracle DP tập con và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle DP tập con và 9 cài đặt đúng


### B. Edge case

- Một viên: `0`.
- Hai viên cùng hàng / cùng cột / không chung: `1`, `1`, `0`.
- Mọi viên cùng một hàng (hoặc cột): `n − 1`.
- Mọi viên khác hàng và khác cột (đường chéo): `0`.
- Đường chéo cộng một hàng "nối" tất cả: một thành phần.
- Hàng và cột **trùng số** (`x = y` dễ va chạm): `[0,1]` và `[1,2]`.
- Hai thành phần cách xa nhau.
- Lưới `3 × 3` đầy đủ: `9 − 1 = 8`.

In [17]:
cases = [
    ([[0, 0]], 0), ([[0, 0], [0, 5]], 1), ([[0, 0], [7, 0]], 1), ([[0, 0], [1, 1]], 0),
    ([[0, y] for y in range(6)], 5), ([[x, 3] for x in range(6)], 5), ([[i, i] for i in range(6)], 0),
    ([[i, i] for i in range(5)] + [[0, 4]], None), ([[0, 1], [1, 2]], 0), ([[0, 0], [0, 1], [10, 10], [10, 11]], 2),
    ([[x, y] for x in range(3) for y in range(3)], 8), ([[1, 0], [0, 1]], 0), ([[2, 1], [1, 2], [1, 1]], 2),
]
for st, want in cases:
    check(st, want)
print(len(cases), "edge case: oracle và mọi cài đặt đúng")

13 edge case: oracle và mọi cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def bug_return_components(stones):                     # trả số thành phần thay vì n - số thành phần
    return len({tuple(c) for c in components_pairs(stones)})


def bug_id_collision(stones):                          # hàng và cột dùng chung không gian số nguyên
    parent = {}

    def find(a):
        parent.setdefault(a, a)
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    for x, y in stones:
        ra, rb = find(x), find(y)                      # SAI: x và y cùng một không gian chỉ số
        if ra != rb:
            parent[ra] = rb
    return len(stones) - len({find(x) for x, y in stones})


def bug_rows_only(stones):                             # chỉ nối các đá cùng hàng
    rows = defaultdict(int)
    for x, y in stones:
        rows[x] += 1
    return len(stones) - len(rows)


def bug_distinct_rows_cols(stones):                    # n - (số hàng phân biệt) - (số cột phân biệt) + ...
    xs = {x for x, y in stones}
    ys = {y for x, y in stones}
    return max(0, len(stones) - max(len(xs), len(ys)))


def bug_missing_col_in_union(stones):                  # BFS chỉ duyệt hàng, quên cột
    rows = defaultdict(list)
    for i, (x, y) in enumerate(stones):
        rows[x].append(i)
    seen = [False] * len(stones)
    comps = 0
    for s in range(len(stones)):
        if seen[s]:
            continue
        comps += 1
        stack = [s]
        seen[s] = True
        while stack:
            u = stack.pop()
            for v in rows[stones[u][0]]:
                if not seen[v]:
                    seen[v] = True
                    stack.append(v)
    return len(stones) - comps


def bug_minus_one(stones):                             # n - C - 1 (cho rằng còn lại ít hơn)
    return max(0, remove_stones_dsu(stones) - 1)


BUGS = [bug_return_components, bug_id_collision, bug_rows_only, bug_distinct_rows_cols, bug_missing_col_in_union, bug_minus_one]
rng = random.Random(3)
tests = [random_stones(rng, rng.randint(1, 8), 3, 3) for _ in range(1000)]
for bug in BUGS:
    bad = next((s for s in tests if bug(s) != oracle(s)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:26s} sai trên {sorted(map(tuple, bad))}: ra {bug(bad)}, đúng {oracle(bad)}")

bug_return_components      sai trên [(0, 3), (1, 2), (2, 1), (3, 0)]: ra 4, đúng 0
bug_id_collision           sai trên [(0, 3), (1, 2), (2, 1), (3, 0)]: ra 2, đúng 0
bug_rows_only              sai trên [(1, 1), (3, 0), (3, 1), (3, 3)]: ra 2, đúng 3
bug_distinct_rows_cols     sai trên [(1, 1), (3, 0), (3, 1), (3, 3)]: ra 1, đúng 3
bug_missing_col_in_union   sai trên [(1, 1), (3, 0), (3, 1), (3, 3)]: ra 2, đúng 3
bug_minus_one              sai trên [(1, 1), (3, 0), (3, 1), (3, 3)]: ra 2, đúng 3


### D. Vét cạn mọi tập con của lưới nhỏ

Mọi tập con khác rỗng của lưới `3 × 3` (`511`) và của lưới `3 × 4` (`4 095`). Oracle: DP tập con. Tổng `4 606` tập đá — mọi cấu hình liên thông/không liên thông trên lưới nhỏ.

In [19]:
t0 = time.perf_counter()
cnt = 0
fast = ("dfs", "dsu", "bfs", "sequence", "Solution")
for W, H in [(3, 3), (3, 4)]:
    cells = [[x, y] for x in range(W) for y in range(H)]
    for mask in range(1, 1 << len(cells)):
        st = [cells[i] for i in range(len(cells)) if mask >> i & 1]
        want = oracle(st)
        for name in fast:
            assert IMPLS[name](st) == want, (name, st)
        cnt += 1
print(cnt, "tập đá: 5 cài đặt khớp oracle DP tập con;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 511 + 4095

4606 tập đá: 5 cài đặt khớp oracle DP tập con; 0.5 s


### E. Ngẫu nhiên có seed và tính chất

Dữ liệu ngẫu nhiên tới `12` đá (so oracle) và tới `120` đá (so các cách với nhau). Tính chất: (1) tịnh tiến, hoán vị trục `(x, y) → (y, x)`, hoặc **nén toạ độ** (đổi tên hàng/cột đơn điệu) không đổi đáp án; (2) `0 ≤ đáp án ≤ n − 1`; (3) thêm một viên đá **cùng hàng hoặc cùng cột** với một viên có sẵn (ở ô trống) làm đáp án **tăng ít nhất 1** (tăng đúng 1 nếu nó không nối thêm thành phần khác, nhiều hơn nếu nó hợp nhất thêm thành phần); (4) dãy loại bỏ tường minh hợp lệ (`simulate_removals` qua).

In [20]:
rng = random.Random(20240947)
for _ in range(1200):
    n = rng.randint(1, 12)
    X, Y = rng.choice([(3, 3), (5, 5), (8, 2), (20, 20)])
    st = random_stones(rng, min(n, (X + 1) * (Y + 1)), X, Y)
    want = oracle(st)
    for name in ("dfs", "dsu", "bfs", "sequence", "Solution", "SolutionDFS", "SolutionDSU", "SolutionBFS", "SolutionSequence"):
        assert IMPLS[name](st) == want, (name, st)
    dx, dy = rng.randint(0, 50), rng.randint(0, 50)
    assert Solution().removeStones([[x + dx, y + dy] for x, y in st]) == want                    # (1) tịnh tiến
    assert Solution().removeStones([[y, x] for x, y in st]) == want                               # (1) hoán vị trục
    xs, ys = sorted({x for x, _ in st}), sorted({y for _, y in st})
    mx, my = {v: 3 * i + 1 for i, v in enumerate(xs)}, {v: 5 * i + 2 for i, v in enumerate(ys)}
    assert Solution().removeStones([[mx[x], my[y]] for x, y in st]) == want                       # (1) nén/đổi tên toạ độ đơn điệu
    assert 0 <= want <= len(st) - 1                                                               # (2)
    existing = {tuple(p) for p in st}
    x0, y0 = rng.choice(st)
    if rng.random() < 0.5:
        cand = [[x0, y] for y in range(Y + 1) if (x0, y) not in existing]
    else:
        cand = [[x, y0] for x in range(X + 1) if (x, y0) not in existing]
    if cand:
        assert Solution().removeStones(st + [rng.choice(cand)]) >= want + 1                       # (3) thêm đá cùng hàng/cột
    assert simulate_removals(st, removal_sequence(st))                                            # (4)
for _ in range(100):
    st = random_stones(rng, rng.randint(30, 120), 15, 15)
    assert remove_stones_dfs(st) == remove_stones_dsu(st) == remove_stones_bfs(st) == Solution().removeStones(st) == remove_stones_sequence(st)
print("1200 tập đá ngẫu nhiên khớp oracle; 4 tính chất (bất biến toạ độ, cận, thêm đá, dãy loại hợp lệ) và 100 tập tới 120 đá đúng")

1200 tập đá ngẫu nhiên khớp oracle; 4 tính chất (bất biến toạ độ, cận, thêm đá, dãy loại hợp lệ) và 100 tập tới 120 đá đúng


### F. Giới hạn: `n = 1000`

Các cách khớp nhau trên nhiều dạng dữ liệu: ngẫu nhiên thưa, lưới đặc, một hàng, một cột, đường chéo, nhiều thành phần nhỏ, một thành phần khổng lồ dạng "bậc thang".

In [21]:
rng = random.Random(500)
n = 1000
fast = ["dfs", "dsu", "bfs", "sequence", "Solution", "SolutionDFS", "SolutionDSU", "SolutionBFS", "SolutionSequence"]
datasets = {
    "thưa 10^4": random_stones(rng, n, 10 ** 4, 10 ** 4),
    "lưới đặc 32x32": [[x, y] for x in range(32) for y in range(32)][:n],
    "một hàng": [[5, y] for y in range(n)],
    "một cột": [[x, 5] for x in range(n)],
    "đường chéo": [[i, i] for i in range(n)],
    "bậc thang": [[i, i] for i in range(500)] + [[i, i + 1] for i in range(500)],
    "nhiều thành phần 2 đá": [[2 * i, 2 * i] for i in range(500)] + [[2 * i, 2 * i + 1] for i in range(500)],
}
for name, st in datasets.items():
    res = {k: IMPLS[k](st) for k in fast}
    assert len(set(res.values())) == 1, (name, res)
    print(f"{name:24s}: n = {len(st)}, loại được {res['dsu']}")
assert Solution().removeStones(datasets["một hàng"]) == n - 1 and Solution().removeStones(datasets["đường chéo"]) == 0
assert Solution().removeStones(datasets["bậc thang"]) == 1000 - 1 and Solution().removeStones(datasets["nhiều thành phần 2 đá"]) == 500
print("công thức đóng khớp")

thưa 10^4               : n = 1000, loại được 111
lưới đặc 32x32          : n = 1000, loại được 999


một hàng                : n = 1000, loại được 999


một cột                 : n = 1000, loại được 999


đường chéo              : n = 1000, loại được 0


bậc thang               : n = 1000, loại được 999


nhiều thành phần 2 đá   : n = 1000, loại được 500
công thức đóng khớp


## 13. Các lỗi người mới hay gặp

### Lỗi 1: trả số thành phần thay vì `n − C`

In [22]:
st = examples[1][0]
print("số thành phần:", bug_return_components(st), "| đúng (n - C):", Solution().removeStones(st))
assert bug_return_components(st) == 2 and Solution().removeStones(st) == 3

số thành phần: 2 | đúng (n - C): 3


**Vì sao sai:** mỗi thành phần giữ lại **một** viên; số viên loại được là phần còn lại. **Đúng:** `n − C`.

### Lỗi 2: hàng và cột chung không gian chỉ số

In [23]:
st = [[0, 1], [1, 2]]
print("va chạm chỉ số:", bug_id_collision(st), "| đúng:", Solution().removeStones(st))
assert bug_id_collision(st) == 1 and Solution().removeStones(st) == 0

va chạm chỉ số: 1 | đúng: 0


**Vì sao sai:** `[0,1]` và `[1,2]` không chung hàng cũng không chung cột; nhưng `union(0, 1)` và `union(1, 2)` coi hàng `1` và cột `1` là một nút. **Đúng:** `("r", x)` và `("c", y)` hoặc `y + 10001`.

### Lỗi 3: chỉ nối theo hàng

In [24]:
st = examples[0][0]
print("chỉ hàng:", bug_rows_only(st), "| đúng:", Solution().removeStones(st))
assert bug_rows_only(st) < Solution().removeStones(st)

chỉ hàng: 3 | đúng: 5


**Vì sao sai:** đá cùng **cột** cũng nối nhau; bỏ qua làm thiếu liên thông. **Đúng:** nối theo cả hàng lẫn cột.

### Lỗi 4: đếm số hàng/cột phân biệt thay cho thành phần

In [25]:
bad = next(s for s in tests if bug_distinct_rows_cols(s) != oracle(s))
print(sorted(map(tuple, bad)), "->", bug_distinct_rows_cols(bad), "| đúng:", oracle(bad))

[(1, 1), (3, 0), (3, 1), (3, 3)] -> 1 | đúng: 3


**Vì sao sai:** số hàng/cột phân biệt không bằng số thành phần (nhiều hàng có thể thuộc cùng thành phần, nhiều thành phần có thể có ít hàng). **Đúng:** đếm thành phần liên thông.

### Lỗi 5: BFS quên duyệt cột

In [26]:
bad = next(s for s in tests if bug_missing_col_in_union(s) != oracle(s))
print(sorted(map(tuple, bad)), "->", bug_missing_col_in_union(bad), "| đúng:", oracle(bad))

[(1, 1), (3, 0), (3, 1), (3, 3)] -> 2 | đúng: 3


**Vì sao sai:** thiếu cạnh cùng cột làm số thành phần bị đếm thừa. **Đúng:** duyệt cả `rows[x]` và `cols[y]`.

### Lỗi 6 (bonus): `n − C − 1`

In [27]:
bad = next(s for s in tests if bug_minus_one(s) != oracle(s))
print(sorted(map(tuple, bad)), "->", bug_minus_one(bad), "| đúng:", oracle(bad))

[(1, 1), (3, 0), (3, 1), (3, 3)] -> 2 | đúng: 3


**Vì sao sai:** không cần trừ thêm; mỗi thành phần còn đúng một viên.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "loại bỏ phần tử khi còn phần tử khác cùng hàng/cột (cùng nhóm)" → **đồ thị liên thông**; bài toán thành "đếm thành phần". Mô hình **đồ thị hai phía hàng–cột** gọn hơn đồ thị các điểm.

**Mẹo:**

1. Mỗi viên đá = **cạnh** nối nút hàng và nút cột.
2. DSU với khoá **phân biệt** hàng/cột.
3. Đáp án `n − (số thành phần)`.
4. DFS lặp (hoặc BFS) tránh giới hạn đệ quy.
5. Mỗi thành phần loại được `size − 1`: chứng minh bằng cây khung và thứ tự loại ngược.
6. Oracle: DP trên tập con mô phỏng luật loại bỏ.

**Họ bài cùng pattern:** Number of Provinces (547), Accounts Merge (721), Number of Islands (200), Redundant Connection (684), Connecting Cities With Minimum Cost, Largest Component Size by Common Factor (952), Smallest String With Swaps (1202).

## 15. Biến thể

1. **Số viên còn lại tối thiểu** (`= C`).
2. **Chỉ được loại nếu cùng hàng** (không tính cột): đáp án `n − số hàng phân biệt`.
3. **Số thành phần** (đảo vai).
4. **Chi phí loại khác nhau**: tối đa tổng chi phí khi mỗi thành phần phải giữ lại **một** viên (giữ viên rẻ nhất).
5. **Đồ thị với ba chiều** (hàng, cột, tầng).

Cài biến thể 1, 2, 4 và đối chiếu vét cạn:

In [28]:
def min_remaining(stones):
    return len(stones) - Solution().removeStones(stones)


def removable_row_only(stones):
    return len(stones) - len({x for x, _ in stones})


def oracle_row_only(stones):
    n = len(stones)
    adj = [sum(1 << j for j in range(n) if j != i and stones[i][0] == stones[j][0]) for i in range(n)]
    best = [0] * (1 << n)
    for mask in range(1, 1 << n):
        b, m = 0, mask
        while m:
            low = m & -m
            s = low.bit_length() - 1
            m ^= low
            if adj[s] & mask:
                b = max(b, 1 + best[mask ^ low])
        best[mask] = b
    return best[(1 << n) - 1]


def max_total_weight_removed(stones, weights):
    """Biến thể 4: mỗi thành phần giữ lại viên có TRỌNG SỐ NHỎ NHẤT; tổng trọng số các viên loại tối đa."""
    comps = components_pairs(stones)
    return sum(sum(weights[i] for i in c) - min(weights[i] for i in c) for c in comps)


def oracle_weighted(stones, weights):
    n = len(stones)
    adj = [sum(1 << j for j in range(n) if j != i and (stones[i][0] == stones[j][0] or stones[i][1] == stones[j][1])) for i in range(n)]
    best = [0] * (1 << n)
    for mask in range(1, 1 << n):
        b, m = 0, mask
        while m:
            low = m & -m
            s = low.bit_length() - 1
            m ^= low
            if adj[s] & mask:
                b = max(b, weights[s] + best[mask ^ low])
        best[mask] = b
    return best[(1 << n) - 1]


rng = random.Random(15)
for _ in range(400):
    st = random_stones(rng, rng.randint(1, 9), 3, 3)
    assert min_remaining(st) == len(components_pairs(st))
    assert removable_row_only(st) == oracle_row_only(st)
    w = [rng.randint(1, 9) for _ in st]
    assert max_total_weight_removed(st, w) == oracle_weighted(st, w)
print("còn lại tối thiểu = C; chỉ cùng hàng = n - số hàng; tổng trọng số loại tối đa: khớp oracle DP tập con trên 400 tập đá")

còn lại tối thiểu = C; chỉ cùng hàng = n - số hàng; tổng trọng số loại tối đa: khớp oracle DP tập con trên 400 tập đá


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đồ thị đá + DFS | cạnh cùng hàng/cột, `O(n²)` | `O(n²)` | `O(n)` | oracle thứ hai, `n` nhỏ |
| 2. Union-Find hàng/cột | `union(("r",x), ("c",y))` | `O(n α(n))` | `O(n)` | **đáp án chính** |
| 3. BFS bảng băm | `rows[x]`, `cols[y]` | `O(n)` | `O(n)` | không cần DSU |
| 4. Dãy loại bỏ | DFS ngược, `size − 1` mỗi thành phần | `O(n)` | `O(n)` | chứng minh cận dưới |

Cả bốn khớp oracle DP tập con trên 4 606 tập đá của lưới nhỏ và 1 200 tập ngẫu nhiên, và khớp nhau ở `n = 1000` (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [29]:
print(Solution().removeStones([[0, 0], [0, 1], [1, 0]]), Solution().removeStones([[0, 0], [1, 1], [2, 2]]), Solution().removeStones([[0, 0], [0, 2], [1, 1], [2, 0], [2, 2], [5, 5], [5, 6]]), min_remaining([[0, 0], [1, 1], [0, 1]]))

2 0 4 1


### Bài 2 (tính tay)

`stones = [[1,1],[1,3],[3,1],[3,3],[2,2],[4,4],[4,5]]`. (a) Các thành phần? (b) Đáp án? (c) Viên nào còn lại nếu chọn gốc là viên có chỉ số nhỏ nhất mỗi thành phần? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện Union-Find:

In [30]:
def dsu_todo(stones):
    parent = {}

    def find(a):
        parent.setdefault(a, a)
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    for x, y in stones:
        pass            # TODO 1: union(("r", x), ("c", y))
    return 0            # TODO 2: len(stones) - số thành phần


# Khi làm xong, bỏ comment để kiểm tra:
# assert [dsu_todo(st) for st, _ in examples] == [5, 3, 0]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **BFS với bảng băm hàng/cột**:

In [31]:
class MySolution:
    def removeStones(self, stones):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().removeStones([list(p) for p in st]) == w for st, w in examples)

### Bài 5 (mini challenge): Number of Provinces (547)

Cho ma trận kề `isConnected` (`n × n`, đối xứng), đếm số **tỉnh** (thành phần liên thông). Viết `find_circle_num(isConnected)` bằng Union-Find và kiểm tra bằng DFS.

In [32]:
def find_circle_num(isConnected):
    pass            # TODO


# Khi làm xong:
# assert find_circle_num([[1, 1, 0], [1, 1, 0], [0, 0, 1]]) == 2 and find_circle_num([[1, 0, 0], [0, 1, 0], [0, 0, 1]]) == 3

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: đáp án `n − số thành phần`.
- Mức 2: `[[0,0],[0,1],[1,0]]`: cả ba nối nhau (một thành phần) ⇒ `2`; `[[0,0],[1,1],[2,2]]`: ba thành phần ⇒ `0`.
- Mức 3: kết quả `2 0 4 1`: ví dụ thứ ba có ba thành phần (bốn đá góc, `[1,1]`, và `[5,5]`–`[5,6]`) nên `7 − 3 = 4`; `min_remaining([[0,0],[1,1],[0,1]]) = 1` (ba đá nối thành một thành phần).

**Bài 2**

- Mức 1: bốn đá `[1,1],[1,3],[3,1],[3,3]` nối nhau theo hàng/cột.
- Mức 2: `[2,2]` đứng một mình; `[4,4],[4,5]` cùng hàng `4`.
- Mức 3: ba thành phần ⇒ `7 − 3 = 4`; còn lại: `[1,1]`, `[2,2]`, `[4,4]`.

**Bài 3**

- Mức 1: `parent[find(("r", x))] = find(("c", y))`.
- Mức 2: số thành phần = số gốc khác nhau của `find(("r", x))`.
- Mức 3: `return len(stones) - comps`.

**Bài 4**

- Mức 1: `rows`, `cols` bằng `defaultdict(list)`.
- Mức 2: BFS; đánh dấu hàng/cột đã dùng.
- Mức 3: `return n - comps`.

**Bài 5**

- Mức 1: DSU trên `n` đỉnh; `union(i, j)` nếu `isConnected[i][j]`.
- Mức 2: số tỉnh = số gốc khác nhau.
- Mức 3: kiểm tra bằng DFS.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [33]:
vals = (Solution().removeStones([[0, 0], [0, 1], [1, 0]]), Solution().removeStones([[0, 0], [1, 1], [2, 2]]),
        Solution().removeStones([[0, 0], [0, 2], [1, 1], [2, 0], [2, 2], [5, 5], [5, 6]]), min_remaining([[0, 0], [1, 1], [0, 1]]))
assert vals == (2, 0, 4, 1) == (oracle([[0, 0], [0, 1], [1, 0]]), oracle([[0, 0], [1, 1], [2, 2]]),
                               oracle([[0, 0], [0, 2], [1, 1], [2, 0], [2, 2], [5, 5], [5, 6]]), 1)
print("Bài 1 OK:", vals)

Bài 1 OK: (2, 0, 4, 1)


### Bài 2

(a) `{[1,1],[1,3],[3,1],[3,3]}`, `{[2,2]}`, `{[4,4],[4,5]}`. (b) `7 − 3 = 4`. (c) còn lại `[1,1]`, `[2,2]`, `[4,4]`.

In [34]:
st = [[1, 1], [1, 3], [3, 1], [3, 3], [2, 2], [4, 4], [4, 5]]
comps = components_pairs(st)
assert comps == [[0, 1, 2, 3], [4], [5, 6]]
assert Solution().removeStones(st) == oracle(st) == 4
assert [st[c[0]] for c in comps] == [[1, 1], [2, 2], [4, 4]]
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [35]:
def dsu_answer(stones):
    parent = {}

    def find(a):
        parent.setdefault(a, a)
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    for x, y in stones:
        parent[find(("r", x))] = find(("c", y))
    return len(stones) - len({find(("r", x)) for x, y in stones})


assert [dsu_answer(st) for st, _ in examples] == [5, 3, 0]
for s in tests[:300]:
    assert dsu_answer(s) == oracle(s)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MySolutionAnswer:
    def removeStones(self, stones):
        rows, cols = defaultdict(list), defaultdict(list)
        for i, (x, y) in enumerate(stones):
            rows[x].append(i)
            cols[y].append(i)
        seen, comps = [False] * len(stones), 0
        ur, uc = set(), set()
        for s in range(len(stones)):
            if seen[s]:
                continue
            comps += 1
            q = deque([s])
            seen[s] = True
            while q:
                u = q.popleft()
                x, y = stones[u]
                if x not in ur:
                    ur.add(x)
                    for v in rows[x]:
                        if not seen[v]:
                            seen[v] = True
                            q.append(v)
                if y not in uc:
                    uc.add(y)
                    for v in cols[y]:
                        if not seen[v]:
                            seen[v] = True
                            q.append(v)
        return len(stones) - comps


assert all(MySolutionAnswer().removeStones([list(p) for p in st]) == w for st, w in examples)
for s in tests[:300]:
    assert MySolutionAnswer().removeStones([list(p) for p in s]) == oracle(s)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [37]:
def find_circle_num_answer(isConnected):
    n = len(isConnected)
    parent = list(range(n))

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    for i in range(n):
        for j in range(i + 1, n):
            if isConnected[i][j]:
                parent[find(i)] = find(j)
    return len({find(i) for i in range(n)})


def find_circle_num_dfs(isConnected):
    n, seen, comps = len(isConnected), set(), 0
    for s in range(n):
        if s in seen:
            continue
        comps += 1
        stack = [s]
        seen.add(s)
        while stack:
            u = stack.pop()
            for v in range(n):
                if isConnected[u][v] and v not in seen:
                    seen.add(v)
                    stack.append(v)
    return comps


assert find_circle_num_answer([[1, 1, 0], [1, 1, 0], [0, 0, 1]]) == 2 and find_circle_num_answer([[1, 0, 0], [0, 1, 0], [0, 0, 1]]) == 3
rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 7)
    m = [[1 if i == j else 0 for j in range(n)] for i in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            if rng.random() < 0.25:
                m[i][j] = m[j][i] = 1
    assert find_circle_num_answer(m) == find_circle_num_dfs(m)
print("Bài 5 OK: Union-Find khớp DFS")

Bài 5 OK: Union-Find khớp DFS


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (loại bỏ khi cùng nhóm → đếm thành phần liên thông, đá = cạnh hàng–cột):**

```python
class Solution:
    def removeStones(self, stones: List[List[int]]) -> int:
        parent = {}
        def find(a):
            parent.setdefault(a, a)
            while parent[a] != a:
                parent[a] = parent[parent[a]]
                a = parent[a]
            return a
        for x, y in stones:
            parent[find(("r", x))] = find(("c", y))      # tách không gian hàng / cột
        return len(stones) - len({find(("r", x)) for x, y in stones})
```

**Checklist trước khi submit:**

- [ ] Đáp án là `n − (số thành phần liên thông)` (không phải số thành phần)?
- [ ] Hàng và cột dùng **khoá phân biệt** (`("r", x)`, `("c", y)` hoặc `y + 10001`)?
- [ ] Nối theo **cả hàng và cột**?
- [ ] DFS/BFS lặp (hoặc DSU), không đệ quy sâu?
- [ ] Đã thử 3 ví dụ, một viên, tất cả cùng hàng, đường chéo, `[[0,1],[1,2]]`?
- [ ] Đúng tên `class Solution`, `removeStones(self, stones)`, không in debug.